<center style="padding: 1em 2em;  margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0 0 0.5em 0; color:rgb(52, 72, 92);">
        «Функции: вызов, кадр, имена»
    </h1>
</center>


Сегодня мы будем собирать функцию снизу вверх: сначала как ею пользоваться, потом что происходит при вызове, потом где ищутся имена.

---


## **Сегодня пройдемся по этому плану:**

1. **Основы** – зачем функция, как её объявить и вызвать, как передать аргументы.
2. **Устройство вызова** – объект, кадр, стек, рекурсия.
3. **Имена** – свой кадр, модуль, встроенные. Функция внутри функции – в следующий раз.

Каждый кусок нужен следующему.

---


## **Что такое функция и зачем она.**

Кусок кода можно копировать. Через неделю в пяти местах чуть-чуть разные копии, и баг чинится только в трёх.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Функция:</strong><br>именованный кусок кода, к которому можно обратиться из другого места программы.
</div>

Зачем она, кроме «не копировать»:

- одно место правки;
- программа читается как набор шагов с говорящими именами, а не как простыня;
- имя функции и имена параметров сами документируют намерение.

Синтаксис подождёт один абзац. Сначала вопрос, который без функции неудобен: сравнить два числа относительно их масштаба, а не «на сколько единиц больше».

### **Объявление.**

`def` создаёт функцию. Имя, скобки, двоеточие, тело с отступом.

Если тела ещё нет, а синтаксис уже нужен – в блоке пишут `pass`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>pass</code>:</strong><br>пустая инструкция. «Здесь должен быть блок, но делать пока нечего». Без неё <code>def</code> с пустым телом – синтаксическая ошибка.
</div>


In [ ]:
def function():
    pass  # блок есть, действий нет


### **Вызов и возвращаемое значение.**

Имя со скобками – просьба выполнить. Результат можно положить в переменную.

В Python функция **всегда** что-то возвращает. Нет `return` – возвращается `None`.


In [ ]:
result = function()
print(result)  # -> None


### **Осмысленный пример.**

Считаем относительную разницу: насколько `x` и `y` разъехались по сравнению со своим средним. Один вопрос – одна функция.


In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    return abs(delta / mean)


print(relative_difference(3, 5))  # -> 0.5


### **Аргументы: по позиции и по имени.**

При вызове значения подставляются в параметры. По порядку – позиционно. По имени – тогда порядок не важен.

> **PEP 8.** Вокруг `=` при передаче по имени пробелы не ставят: `y=5`, не `y = 5`.


In [ ]:
print(relative_difference(3, 5))      # оба позиционно
print(relative_difference(y=5, x=3))  # оба по имени, порядок свободный
print(relative_difference(3, y=5))    # сначала позиция, потом имя


Имя, а потом позиция – нельзя. Python уже не знает, какой слот ещё свободен.


In [ ]:
# relative_difference(x=3, 5)  # SyntaxError: positional argument follows keyword argument


Та же функция ломается, если среднее равно нулю: `x` и `-x`. Делить на ноль нельзя – Python бросает `ZeroDivisionError`.


In [ ]:
# relative_difference(5, -5)  # ZeroDivisionError: float division by zero


### **Несколько `return`. Вызов заканчивается сразу.**

Сначала поправим в лоб: посчитали среднее, если оно почти ноль – кладём `None` в переменную, иначе – отношение. Один `return` в конце.


In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        res = None
    else:
        res = abs(delta / mean)
    return res


print(relative_difference(5, -5))  # -> None
print(relative_difference(3, 5))   # -> 0.5


Тот же смысл короче: функция может содержать несколько `return`. Первый сработавший *заканчивает* вызов, дальше тело не идёт.


In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        return None  # вызов закончен, до нижней строки не дойдём
    return abs(delta / mean)


print(relative_difference(5, -5))  # -> None
print(relative_difference(3, 5))   # -> 0.5


### **Документация и аннотации.**

Сразу под `def` – строка в тройных кавычках. Это `__doc__`. Её печатает `help`.

После параметров и после `->` – подсказки типов. Python их **не проверяет** в рантайме. Это записка для людей, `help` и тайпчекера.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>help(объект)</code>:</strong><br>печатает справку: подпись, документацию. Особенно удобно в интерактивном режиме.
</div>


In [ ]:
def relative_difference(x: float, y: float) -> float | None:
    """Сравнивает две величины с учётом их масштаба."""
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        return None
    return abs(delta / mean)


help(relative_difference)
help(pow)  # то же самое работает для встроенных функций


Подсказка `int` не мешает передать строку. Ошибки на вызове не будет – тип никто не сверяет.


In [ ]:
def identity(value: int) -> int:
    return value


print(identity("123"))  # print рисует 123, это по-прежнему строка


### **Значение по умолчанию.**

Параметр можно не передавать, если у него есть запас. Запас пишут в `def` после `=`.

> **PEP 8.** Если у параметра есть и тип, и дефолт, вокруг `=` *ставят* пробелы: `verbose: bool = False`.


In [ ]:
def relative_difference(
    x: float,
    y: float,
    verbose: bool = False,
) -> float | None:
    delta = x - y
    if verbose:
        print(f"delta: {delta}")
    mean = (x + y) / 2
    if verbose:
        print(f"mean: {mean}")
    if abs(mean) < 1e-10:
        return None
    return abs(delta / mean)


print(relative_difference(-10, 3))
print(relative_difference(-10, 3, verbose=True))


### **Функция – объект.**

`def` кладёт в имя не «команду», а объект. Без скобок мы говорим про объект. Со скобками – про результат вызова.

Поэтому объект можно положить в другое имя и передать в другую функцию.


In [ ]:
def meaning_of_life() -> int:
    return 42


answer = meaning_of_life  # без скобок: второе имя той же функции
print(answer())                  # -> 42
print(answer is meaning_of_life)  # -> True


Локальные имена после вызова уже недоступны. Они жили только *во время* выполнения – к этому вернёмся, когда появится кадр.

Пока достаточно факта: снаружи имени из тела нет.


In [ ]:
def some_function() -> int:
    inside = 15
    return inside


print(some_function())  # -> 15
# print(inside)  # NameError: имени inside снаружи нет


Раз функция – объект, её тип в подсказке – «вызываемое».

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>Callable[[типы аргументов], тип возврата]</code>:</strong><br>объект, который можно вызвать с такими аргументами и получить такой результат.
</div>

Так мы передаём *правило нормировки* в `relative_difference`, а не зашиваем среднее навсегда.


In [ ]:
from collections.abc import Callable


def relative_difference(
    x: float,
    y: float,
    normalize_by: Callable[[float, float], float],
) -> float | None:
    delta = x - y
    norm = normalize_by(x, y)
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


print(relative_difference(-5, 8, normalize_by=max))  # -> 1.625


Нормировку можно не передавать. Тогда параметр опционален: запас `None`, внутри проверяем.

> **PEP 8.** С `None` сравнивают через `is` / `is not`: это один-единственный объект, не «похожее значение».
> Предпочтительнее `is not None`, чем `not ... is None`.


In [ ]:
def relative_difference(
    x: float,
    y: float,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    if normalize_by is not None:
        norm = normalize_by(x, y)
    else:
        norm = 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


def mean(x: float, y: float) -> float:
    return (x + y) / 2


print(relative_difference(-5, 8))                     # -> 13.0, норма 1
print(relative_difference(-5, 8, normalize_by=mean))  # -> ~8.67
print(relative_difference(-5, 8, mean))               # то же позиционно


### **Только по имени: `*`.**

Позиционно `mean` легко перепутать с третьим числом. Звёздочка в списке параметров говорит: всё *правее* передаётся только по имени.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Именованный только (<code>*</code>):</strong><br>параметры после звёздочки нельзя передать позиционно.
</div>


In [ ]:
def relative_difference(
    x: float,
    y: float,
    *,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    norm = normalize_by(x, y) if normalize_by is not None else 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


# relative_difference(-5, 8, mean)  # TypeError: takes 2 positional arguments but 3 were given
print(relative_difference(-5, 8, normalize_by=mean))


### **Только позиционно: `/`.**

Обратное ограничение: параметры *левее* слэша нельзя передать по имени. Так устроены некоторые встроенные: имя параметра – деталь реализации, не часть контракта.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Позиционный только (<code>/</code>):</strong><br>параметры до слэша передаются только по порядку.
</div>


In [ ]:
def relative_difference(
    x: float,
    y: float,
    /,
    *,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    norm = normalize_by(x, y) if normalize_by is not None else 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


print(relative_difference(-5, 8, normalize_by=mean))
# relative_difference(-5, y=8, normalize_by=mean)
# TypeError: positional-only arguments passed as keyword arguments: 'y'


### **Сколько угодно позиционных: `*args`.**

Пока числа удобно собрать в список и передать одним аргументом.


In [ ]:
def root_mean_square(values: list[float]) -> float:
    if not values:
        return 0.0
    mean = sum(x ** 2 for x in values) / len(values)
    return mean ** 0.5


print(root_mean_square([4, 8, 15, 16, 23, 42]))  # -> ~21.81


Вызов `root_mean_square(4, 8, 15)` так не сделать: функция ждёт *один* список. Звёздочка перед именем собирает «лишние» позиционные в кортеж – скобки вокруг списка в вызове больше не нужны.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>*args</code>:</strong><br>кортеж позиционных аргументов, которые не попали в обычные параметры.
</div>

> **PEP 8.** В аннотации `*args` указывают тип *одного элемента*, не всего кортежа: `*args: float`.


In [ ]:
def root_mean_square(*args: float) -> float:
    if not args:
        return 0.0
    mean = sum(x ** 2 for x in args) / len(args)
    return mean ** 0.5


print(root_mean_square(4, 8, 15, 16, 23, 42))  # -> ~21.81


### **Сколько угодно имён: `**kwargs`.**

Две звёздочки собирают лишние именованные в словарь.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>**kwargs</code>:</strong><br>словарь <code>имя → значение</code> для именованных аргументов, которые не попали в обычные параметры.
</div>


In [ ]:
def root_mean_square(*args: float, **kwargs: object) -> float:
    verbose = bool(kwargs.get("verbose", False))
    if not args:
        if verbose:
            print("пустой список")
        return 0.0
    mean = sum(x ** 2 for x in args) / len(args)
    if verbose:
        print(f"mean square: {mean}")
    return mean ** 0.5


print(root_mean_square(4, 8, 15, 16, 23, 42, verbose=True))


### **Распаковка: звёздочки в вызове.**

В объявлении `*` собирает. В вызове – наоборот, разбрасывает: список в позиционные, словарь в именованные.


In [ ]:
def show(x, y, /, *, option1=None, option2=None) -> None:
    print(x, y, option1, option2)


positional = [4, 8]
key_value = {"option1": 15, "option2": 16}
show(*positional, **key_value)  # -> 4 8 15 16
# show(4, 8, option1=3, **key_value)  # TypeError: multiple values for argument 'option1'


Смешаем всё в одной подписи и посмотрим, куда падает «лишнее». Файлы здесь не читаем – смотрим только на аргументы. `True` без имени не станет `binary=True`: после `*files` позиционное значение уезжает в кортеж.


In [ ]:
def read_and_merge_files(
    base_path: str,
    /,
    *files: str,
    binary: bool = False,
    **kwargs: object,
) -> None:
    print(f"{base_path=}, {files=}, {binary=}, {kwargs=}")


read_and_merge_files("a", "b", "c", "d")
read_and_merge_files("a", "b", "c", "d", True)          # True попал в files
read_and_merge_files("a", "b", "c", "d", binary=True)   # так задумано
read_and_merge_files("a", "b", "c", "d", base_path="x")  # лишний base_path в kwargs


### **`lambda`.**

Тот же объект функции, только без `def` и без имени. Тело – одно выражение, не блок.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #f59e0b; background-color: #fffbf0; color: #92400e;">
<strong>⚠️ Предупреждение:</strong><br>Сохранять лямбду в переменную: тогда нужен обычный <code>def</code> с именем, документацией и типами.
</div>

Место лямбды – короткий ключ сортировки, который больше нигде не нужен.


In [ ]:
print(sorted(["привет", "как", "дела"], key=lambda s: len(s)))  # -> ['как', 'дела', 'привет']


def string_length(s: str) -> int:
    return len(s)


Дальше вопрос не «как вызвать», а *что происходит*, когда мы вызываем.

---


## **Устройство вызова: объект, кадр, стек.**

Локальные имена пропадают после `return`. Куда они делись? Они жили в **кадре** – одном незавершённом вызове.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Объект функции:</strong><br>то, что создаёт <code>def</code>: тело, имена параметров, значения по умолчанию.<br><strong>Кадр (frame):</strong><br>одно выполнение этой функции. Появляется на вызове, исчезает на <code>return</code>.<br><strong>Стек вызовов:</strong><br>стопка кадров. Верхний выполняется сейчас. Вызов кладёт кадр, <code>return</code> снимает.
</div>

Один кадр = один вызов, который ещё не закончился. Два вызова одной функции – два кадра, один объект.

<img src="../../Other/images/func_stacks.png" alt="Стек вызовов" width="320">

Низ стопки – кто всех позвал. На картинке это `main`, в Jupyter тот же слой называется `<module>`. Верх – кадр, который крутится сейчас.


In [ ]:
def add(a, b=10):
    c = a + b
    return c


print(add.__code__.co_varnames)  # -> ('a', 'b', 'c')  локальные имена
print(add.__defaults__)          # -> (10,)  запас на объекте, один на всех


`__code__` – скомпилированное тело. `co_varnames` – локальные имена (сначала аргументы). `__defaults__` лежит *на объекте*, не в кадре.

`inspect.stack()` показывает ту же стопку, что на картинке: нулевой элемент – верх, кадр, который выполняется сейчас.


In [ ]:
import inspect


def names_on_stack():
    return [info.function for info in inspect.stack()]


def bottom():
    print(names_on_stack()[:8])


def middle():
    bottom()


def top():
    middle()


top()
# -> ['names_on_stack', 'bottom', 'middle', 'top', ... и служебные кадры Jupyter]


### **Рекурсия – тот же объект, новые кадры.**

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Рекурсия:</strong><br>функция вызывает сама себя. Отдельного синтаксиса нет: на стек кладётся ещё один кадр той же функции.
</div>

У каждого кадра свои локальные имена. `n` в одном вызове не затирает `n` в другом.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Базовый случай:</strong><br>ветка без рекурсивного вызова. Без неё стек растёт до <code>RecursionError</code>.
</div>


In [ ]:
def countdown(n):
    print("вход", n)
    if n == 0:
        print("дно")
        return
    countdown(n - 1)
    print("выход", n)


countdown(3)
# вход растит стек, выход снимает кадры


Тот же стек, но кадр ещё и *отдаёт значение наверх*: `factorial(5)` ждёт `factorial(4)`, тот – `factorial(3)`, пока не сработает базовый случай.


In [ ]:
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)


print(factorial(5))  # -> 120


`sys.getrecursionlimit()` – потолок числа кадров, не «высота нашей задачи». `sys.setrecursionlimit` как решение не используем: если глубина растёт линейно со входом, обычно нужен цикл.


In [ ]:
import sys

print(sys.getrecursionlimit())  # -> типично 1000


### **Почему `list=[]` в дефолте – один список на всех.**

Два имени могут смотреть на один список. `append` через второе имя меняет тот же объект.


In [ ]:
first = [1, 2, 3]
second = first
second.append(4)
print(first)   # -> [1, 2, 3, 4]
print(second)  # -> [1, 2, 3, 4]


С дефолтом функции – та же ссылка, только создаётся она один раз, на `def`. Кадр при вызове берёт эту ссылку, а не новый список. Все вызовы без аргумента смотрят на один список.


In [ ]:
def add_hi(items=[]):
    items.append("Hi!")
    return items


print(add_hi())             # -> ['Hi!']
print(add_hi())             # -> ['Hi!', 'Hi!']
print(add_hi.__defaults__)  # тот же список лежит на объекте


Чинят так: запас – `None`, список создают *внутри кадра*.


In [ ]:
def add_hi(items=None):
    if items is None:
        items = []
    items.append("Hi!")
    return items


print(add_hi())  # -> ['Hi!']
print(add_hi())  # -> ['Hi!']


Если `None` – законное значение «ничего не делать», дефолт должен быть другим объектом.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Sentinel:</strong><br>отдельный объект «значение не передали». Обычно <code>object()</code>. Нужен, когда <code>None</code> уже значит что-то своё.
</div>


In [ ]:
missing = object()


def add_hi(items=missing):
    if items is None:
        return None  # явно сказали «ничего»
    if items is missing:
        items = []  # вызвали без аргумента
    items.append("Hi!")
    return items


print(add_hi())      # -> ['Hi!']
print(add_hi(None))  # -> None


## **Где живут имена.**

Таблица имён вызова – поля кадра. Цикл и `if` кадр не создают: они крутятся в том же кадре, поэтому переменная из `for` остаётся снаружи.

Поиск имени идёт по маршруту `LEGB`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>LEGB</code>:</strong><br>Local (текущий кадр) → Enclosing (кадр внешней функции) → Global (модуль) → Built-in (<code>len</code>, <code>print</code>).
</div>

<img src="../../Other/images/python_namespace.png" alt="Пространства имён LEGB" width="420">

Ищем из центра наружу. Сегодня проходим свой кадр, модуль и встроенные. Enclosing – кадр *другой* функции, внутри которой мы объявлены. Такой функции у нас ещё нет: она появится в следующий раз.


In [ ]:
name = "снаружи"

for name in range(3):
    in_for = name

print(in_for)  # -> 2
print(name)    # -> 2, строка потерялась: for не создал кадр


Своего имени в кадре нет – идём дальше. `label` находится в модуле, `len` – среди встроенных.


In [ ]:
label = "global"


def show():
    print(label)  # своего label нет – берём из модуля


show()
print(len([1, 2, 3]))  # built-in


Запись в имя делает его локальным на весь кадр, даже если справа мы ещё хотели прочитать модуль.


In [ ]:
counter = 0


def bump_broken():
    counter = counter + 1  # UnboundLocalError: локальный counter ещё не создан


# bump_broken()


Чтобы писать в модуль, это говорят явно: `global`.


In [ ]:
counter = 0


def bump():
    global counter
    counter = counter + 1
    return counter


print(bump())  # -> 1
print(bump())  # -> 2
print(counter)  # -> 2, имя модуля


Писать в модуль из функции обычно не надо: значение живёт дольше, чем вызов, и его легко затереть из другого места. Зачем тогда внешнее имя, которое *не* модуль, – когда функция объявлена внутри функции.

---


## **Дополнительно почитать.**

Не обязательно до следующей пары.

- [Устройство CPython](https://habr.com/ru/companies/yandex/articles/511972/) – русским текстом про объект кода, кадр и стек. Дальше по статье (память, генераторы, `GIL`) – другие пары. Доклад 2020 года: с 3.11 кадр в CPython устроен иначе, идея та же.
- [Function objects](https://docs.python.org/3/reference/datamodel.html#function-objects) в data model – официально: чем объект функции отличается от объекта кода и от кадра. Дефолты живут на функции, не в коде. Рядом на той же странице – [code objects](https://docs.python.org/3/reference/datamodel.html#code-objects) и [frame objects](https://docs.python.org/3/reference/datamodel.html#frame-objects).

На следующей паре – функция внутри функции: откуда она берёт имена и как из этого получается декоратор.
